In [0]:
# ---------------------------------------------
# PURPOSE:
# Load engineered dataset for evaluation phase
# ---------------------------------------------

from pyspark.sql.functions import col
from pyspark.ml.evaluation import RegressionEvaluator

data_ml = spark.read.parquet(
    "/Volumes/workspace/default/taxi_data/processed_data"
)

print("Total Rows:", data_ml.count())
data_ml.printSchema()


In [0]:
# ---------------------------------------------
# PURPOSE:
# Perform time-aware split
# Train: Jan–Oct
# Test : Nov–Dec
# ---------------------------------------------

from pyspark.sql.functions import month

raw_df = spark.read.parquet("/Volumes/workspace/default/taxi_data/*.parquet")

raw_df = raw_df.filter(col("fare_amount") > 0)

raw_df = raw_df.withColumn("pickup_month", month("tpep_pickup_datetime"))

train_df = raw_df.filter(col("pickup_month") <= 10)
test_df  = raw_df.filter(col("pickup_month") > 10)

print("Train Rows:", train_df.count())
print("Test Rows :", test_df.count())


In [0]:
# ---------------------------------------------
# PURPOSE:
# Reduce skewness using log transformation
# ---------------------------------------------

from pyspark.sql.functions import log1p

train_df = train_df.withColumn("label_log", log1p(col("fare_amount")))
test_df  = test_df.withColumn("label_log", log1p(col("fare_amount")))


In [0]:
# ---------------------------------------------
# PURPOSE:
# Custom Transformer to compute trip duration
# ---------------------------------------------

from pyspark.ml import Transformer
from pyspark.sql.functions import unix_timestamp

class TripDurationTransformer(Transformer):
    
    def _transform(self, dataset):
        return dataset.withColumn(
            "trip_duration",
            unix_timestamp("tpep_dropoff_datetime") - 
            unix_timestamp("tpep_pickup_datetime")
        )

duration_transformer = TripDurationTransformer()

train_df = duration_transformer.transform(train_df)
test_df  = duration_transformer.transform(test_df)


In [0]:
# ---------------------------------------------
# PURPOSE:
# Compare Spark ML vs Scikit-learn
# Using 1% sample
# ---------------------------------------------

sample_df = data_ml.sample(fraction=0.01, seed=42)

pandas_df = sample_df.toPandas()

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np

X = np.vstack(pandas_df["features"].values)
y = pandas_df["label"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

rf_sklearn = RandomForestRegressor(n_estimators=50, max_depth=8)
rf_sklearn.fit(X_train, y_train)

pred = rf_sklearn.predict(X_test)

print("Sklearn R2:", r2_score(y_test, pred))
print("Sklearn MAE:", mean_absolute_error(y_test, pred))
print("Sklearn RMSE:", np.sqrt(mean_squared_error(y_test, pred)))


In [0]:
# ---------------------------------------------
# PURPOSE:
# Extract Feature Importance from Spark RF
# ---------------------------------------------

from pyspark.ml.regression import RandomForestRegressor

rf = RandomForestRegressor(
    labelCol="label",
    featuresCol="features",
    numTrees=40,
    maxDepth=8
)

model = rf.fit(data_ml.sample(0.1))

importances = model.featureImportances.toArray()

feature_names = [
    "passenger_count",
    "trip_distance",
    "RatecodeID",
    "PULocationID_idx",
    "DOLocationID_idx",
    "payment_type_idx",
    "trip_duration",
    "pickup_hour"
]

importance_df = spark.createDataFrame(
    list(zip(feature_names, importances)),
    ["Feature", "Importance"]
)

importance_df.show()


In [0]:
# ---------------------------------------------
# PURPOSE:
# Statistical significance using bootstrap
# ---------------------------------------------

import random

def bootstrap_rmse(df, model, n_iterations=20):
    rmse_values = []
    evaluator = RegressionEvaluator(
        labelCol="label",
        predictionCol="prediction",
        metricName="rmse"
    )
    
    for i in range(n_iterations):
        sample = df.sample(withReplacement=True, fraction=0.2)
        pred = model.transform(sample)
        rmse = evaluator.evaluate(pred)
        rmse_values.append(rmse)
    
    return np.percentile(rmse_values, [2.5, 97.5])

ci = bootstrap_rmse(data_ml.sample(0.2), model)

print("95% RMSE Confidence Interval:", ci)


In [0]:
# ---------------------------------------------
# PURPOSE:
# Measure training time with different partitions
# ---------------------------------------------

import time

for partitions in [50, 100, 200]:
    temp_df = data_ml.repartition(partitions)
    
    start = time.time()
    model = rf.fit(temp_df.sample(0.2))
    duration = time.time() - start
    
    print(f"Partitions: {partitions}, Training Time: {duration}")


In [0]:
from pyspark.ml.linalg import VectorUDT
from pyspark.sql.functions import udf, col
from pyspark.sql.types import ArrayType, DoubleType

# -------------------------------
# Step 1: Convert features to array
# -------------------------------
vector_to_array = udf(lambda v: v.toArray().tolist() if v else [], ArrayType(DoubleType()))
data_ml_array = data_ml.withColumn("features_array", vector_to_array("features"))

# -------------------------------
# Step 2: Flatten array into multiple columns
# -------------------------------
# Get number of features
num_features = len(data_ml_array.select("features_array").first()[0])

# Create column names for features
feature_cols = [f"feature_{i}" for i in range(num_features)]

# Select and expand features
flattened_df = data_ml_array.select(
    *[col("features_array")[i].alias(feature_cols[i]) for i in range(num_features)],
    *[c for c in data_ml_array.columns if c != "features" and c != "features_array"]
)

# -------------------------------
# Step 3: Sample and write to CSV
# -------------------------------
flattened_df.sample(0.05) \
    .write.mode("overwrite") \
    .csv("/Volumes/workspace/default/taxi_data/tableau_dataset", header=True)

print(f"Saved flattened CSV with {num_features} feature columns.")
